# Stage 7：SFT-7B → 无 LACS / LACS 两组训练

本 notebook 与 Jetson Stage 6 A/B 独立，可并行运行。默认先对两组各短测 8 步，再从同一 SFT 起点各正式训练 1 epoch；短测不作为正式起点。保存到 `MyDrive/COMLLM/s7_runs/`。

两组只改变未来成本权重 λ=0 / 0.3，统一使用原 COMLLM 的格式奖励、K=3 蒙特卡洛、G=4、相同种子和数据。原作者的无 LACS 脚本还有不同格式奖励和 G=8，因此这里是**受控的 LACS 消融复现**。训练使用 bf16 LoRA + 本仓库手写 GRPO；原版为 NF4 QLoRA + TRL，不宣称逐字复现原版数值。训练数据为去除全部测试集重合后的 830 条，验证 100 条。

上游固定版本：[COMLLM-MEC-Task-Offloading / 17adedef](https://github.com/Cassandra-Cheng/COMLLM-MEC-Task-Offloading/tree/17adedef587d5dbc02c167d147692a61179cdf4b)。两组正式输出为 `nolacs/lora`、`lacs/lora`；每组结束后预测三套完整测试集（3010+2408+245 条）。

建议当前 96GB G4 或 A100 80GB；不在 Jetson 运行本 notebook。每组单次最多 120 分钟，超时会保存并停止；重新“全部运行”从检查点恢复，不把未完成状态写成完成。

## 1. 环境
保留 Colab 已安装的 PyTorch/CUDA；仅在缺少或版本不符时补齐此前 S3 已用的依赖。

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess, sys, json, os, shutil, hashlib, zipfile, time
from pathlib import Path
required = {"transformers": "5.20.2", "peft": "0.21.2"}
install = []
for name, version in required.items():
    try:
        actual = metadata.version(name)
    except metadata.PackageNotFoundError:
        actual = None
    if actual != version:
        install.append(f"{name}=={version}")
for name in ("accelerate", "sentencepiece", "numpy"):
    if importlib.util.find_spec(name) is None:
        install.append(name)
if install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install], check=True)
# 与 S3 相同：旧版可选 torchao 会阻止 peft 导入；本实验不使用它。
if importlib.util.find_spec("torchao") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "需要支持 bf16 的 CUDA GPU"
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory >= 60 * 1024**3, "本 bf16 LoRA 配置需要至少 60GiB GPU；使用 G4 96GB 或 A100 80GB"
print("GPU:", gpu.name, "GiB:", round(gpu.total_memory/1024**3, 1), "Torch:", torch.__version__)
print({k: metadata.version(k) for k in ("transformers", "peft", "accelerate", "numpy")})


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition GiB: 95.0 Torch: 2.14.1+cu130
{'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0', 'numpy': '2.4.6'}


## 2. 连接 Drive 与参数
默认两组都做。`FULL_MINUTES` 是单组每次调用的时间预算，可恢复；短测和正式输出相互独立。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/COMLLM")
CODE_DRIVE = DRIVE_ROOT / "s7_code"
BASE_DRIVE = DRIVE_ROOT / "reference_model" / "qwen2.5-7b-mec-sft-final-all"
OUT = DRIVE_ROOT / "s7_runs"
WORK = Path("/content/comllm_s7")
BASE = Path("/content/s7_sft7b")
GROUPS = ["nolacs", "lacs"]
SMOKE_STEPS = 8
FULL_MINUTES = 120
OUT.mkdir(parents=True, exist_ok=True)
print("模型来源:", BASE_DRIVE)
print("结果:", OUT)


Mounted at /content/drive
模型来源: /content/drive/MyDrive/COMLLM/reference_model/qwen2.5-7b-mec-sft-final-all
结果: /content/drive/MyDrive/COMLLM/s7_runs


## 3. 核对并展开代码与数据
校验 Drive 上的包大小和 SHA-256，再逐项核对包内清单。权重单独存放，不重复下载网络模型。

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

archive = CODE_DRIVE / "s7_assets.zip"
meta = json.loads((CODE_DRIVE / "s7_assets.meta.json").read_text())
assert archive.stat().st_size == meta["bytes"], "代码包大小不一致"
assert sha256(archive) == meta["sha256"], "代码包 SHA-256 不一致"
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None, "ZIP CRC 失败"
    for name in z.namelist():
        assert not Path(name).is_absolute() and ".." not in Path(name).parts
    z.extractall(WORK)
manifest = json.loads((WORK / "s7_assets_manifest.json").read_text())
for rel, record in manifest["files"].items():
    p = WORK / rel
    assert p.stat().st_size == record["bytes"] and sha256(p) == record["sha256"], rel
assert (WORK / "train/s7_train.py").is_file(), "包里缺少训练入口"
print("包校验通过:", meta["sha256"], "成员:", len(manifest["files"]))
print("数据行数:", {p: v["rows"] for p, v in manifest["data_audit"]["files"].items()})
os.chdir(WORK)


包校验通过: 043439d6d438311d79e5772cba520a4fb44133bd5beda19b5db2f3ced551ecd5 成员: 61
数据行数: {'data/s4/grpo_train.jsonl': 830, 'data/s4/grpo_val.jsonl': 100, 'data/s3/friend_clean.jsonl': 245, 'data/s3/test_seen.jsonl': 3010, 'data/s3/test_unseen.jsonl': 2408}


## 4. 复制并校验 SFT-7B
四个分片逐一核对本机预先计算的 SHA-256。浏览器仍在上传时最多等待 45 分钟；上传完成后自动继续校验、短测和正式训练。


In [ ]:
model_manifest = json.loads((WORK / "model_manifest.json").read_text())
BASE.mkdir(parents=True, exist_ok=True)
# 浏览器上传可与本 notebook 并行；最多等 45 分钟，齐全后自动继续。
wait_deadline = time.monotonic() + 45 * 60
last_report = 0
while True:
    pending = [r["file"] for r in model_manifest["shards"]
               if not (BASE_DRIVE / r["file"]).exists()
               or (BASE_DRIVE / r["file"]).stat().st_size != r["bytes"]]
    if not pending:
        break
    assert time.monotonic() < wait_deadline, f"等待上传超时，仍缺少: {pending}；上传完成后重跑本格"
    if time.monotonic() - last_report >= 60:
        print("等待浏览器上传：", pending, "；完成后自动校验并继续训练", flush=True)
        last_report = time.monotonic()
    time.sleep(15)
need = sum(x["bytes"] for x in model_manifest["shards"] if not (BASE / x["file"]).exists())
assert shutil.disk_usage(BASE).free > need + 5*1024**3, "Colab 本地磁盘空间不足"
for source in (WORK / "model_metadata").iterdir():
    if source.is_file():
        shutil.copy2(source, BASE / source.name)
for record in model_manifest["shards"]:
    name, expected = record["file"], record["sha256"]
    source, target = BASE_DRIVE / name, BASE / name
    assert source.exists(), f"权重还未上传完成: {source}"
    assert source.stat().st_size == record["bytes"], f"Drive 权重大小错误: {name}"
    valid = target.exists() and target.stat().st_size == record["bytes"] and sha256(target) == expected
    if not valid:
        temporary = target.with_suffix(target.suffix + ".partial")
        print("复制:", name, flush=True)
        shutil.copyfile(source, temporary)
        assert sha256(temporary) == expected, f"权重 SHA-256 错误: {name}"
        os.replace(temporary, target)
    print("SHA-256 通过:", name, flush=True)
print("SFT-7B 校验完成；参数:", model_manifest["total_parameters"])


复制: model-00001-of-00004.safetensors
SHA-256 通过: model-00001-of-00004.safetensors
复制: model-00002-of-00004.safetensors
SHA-256 通过: model-00002-of-00004.safetensors
复制: model-00003-of-00004.safetensors
SHA-256 通过: model-00003-of-00004.safetensors
复制: model-00004-of-00004.safetensors
SHA-256 通过: model-00004-of-00004.safetensors
SFT-7B 校验完成；参数: 7615616512


## 5. 数据与身份预检查
预检查不创建 GPU 模型。两组设置、数据、SFT 权重、奖励与训练代码身份不一致时拒绝覆盖已有实验。

In [ ]:
COMMON = [sys.executable, "-u", "-m", "train.s7_train",
          "--base", str(BASE), "--s3-data", str(WORK / "data/s3"),
          "--s4-data", str(WORK / "data/s4"), "--out", str(OUT)]
for group in GROUPS:
    result = subprocess.run(COMMON + ["--group", group, "--mode", "full", "--validate-only"],
                            text=True, capture_output=True)
    if result.returncode:
        print(result.stderr[-6000:])
        raise RuntimeError("预检查失败")
    report = json.loads(result.stdout)
    print(group, "λ=", report["lambda_lacs"], "训练/验证=", report["data"]["train"], report["data"]["val"],
          "测试=", report["data"]["eval_rows"], "剩余重合=", report["data"]["remaining_overlap"])


nolacs λ= 0.0 训练/验证= 830 100 测试= {'test_seen': 3010, 'test_unseen': 2408, 'friend_clean': 245} 剩余重合= 0
lacs λ= 0.3 训练/验证= 830 100 测试= {'test_seen': 3010, 'test_unseen': 2408, 'friend_clean': 245} 剩余重合= 0


## 6. 两组短测并验证可恢复
每组 8 个优化步骤，之后重新加载保存的模型、优化器和随机数状态。正式训练仍从原 SFT 重新开始。

In [ ]:
def invoke(group, mode, extra=(), budget=120):
    command = COMMON + ["--group", group, "--mode", mode, "--resume", "--max-minutes", str(budget), *extra]
    print("开始:", group, mode, flush=True)
    result = subprocess.run(command, cwd=WORK)
    folder = OUT / "smoke" / group if mode == "smoke" else OUT / group
    summary_path = folder / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    print("状态:", summary, flush=True)
    if result.returncode == 75:
        raise RuntimeError(f"{group} 到达时间限制或已保存中断状态；重新运行本格可恢复，未声称完成")
    if result.returncode != 0:
        raise RuntimeError(f"{group} 执行失败，返回 {result.returncode}；请查看本格错误")
    expected = "smoke_complete" if mode == "smoke" else "complete"
    assert summary.get("status") == expected, summary
    return summary

for group in GROUPS:
    invoke(group, "smoke", ["--max-steps", str(SMOKE_STEPS)], budget=30)
    invoke(group, "smoke", ["--max-steps", str(SMOKE_STEPS)], budget=15)
print("两组短测及检查点加载通过。")


开始: nolacs smoke
状态: {'step': 8, 'elapsed_train_s': 93.323045912, 'training_complete': True, 'target_steps': 8, 'status': 'smoke_complete', 'group': 'nolacs', 'invocation_seconds': 128.14352979199975}
开始: nolacs smoke
状态: {'step': 8, 'elapsed_train_s': 93.323045912, 'training_complete': True, 'target_steps': 8, 'status': 'smoke_complete', 'group': 'nolacs', 'invocation_seconds': 22.020799636999982}
开始: lacs smoke
状态: {'step': 8, 'elapsed_train_s': 92.62354691099972, 'training_complete': True, 'target_steps': 8, 'status': 'smoke_complete', 'group': 'lacs', 'invocation_seconds': 123.57038599399993}
开始: lacs smoke
状态: {'step': 8, 'elapsed_train_s': 92.62354691099972, 'training_complete': True, 'target_steps': 8, 'status': 'smoke_complete', 'group': 'lacs', 'invocation_seconds': 22.938803518999975}
两组短测及检查点加载通过。


## 7. 两组正式训练 + 完整开环
先无 LACS，后 LACS。到达时间上限会停止，不自动循环续时；后续再次运行本格恢复。每组每 10 步保存，保留最近两代检查点。

In [ ]:
for group in GROUPS:
    invoke(group, "full", budget=FULL_MINUTES)
print("两组正式训练与开环均完成。")


开始: nolacs full
状态: {'step': 208, 'elapsed_train_s': 2502.6666326019936, 'training_complete': True, 'target_steps': 208, 'status': 'complete', 'group': 'nolacs', 'invocation_seconds': 3606.140981988}
开始: lacs full
状态: {'step': 208, 'elapsed_train_s': 2503.4651944330008, 'training_complete': True, 'target_steps': 208, 'status': 'complete', 'group': 'lacs', 'invocation_seconds': 3608.256021806}
两组正式训练与开环均完成。


## 8. 结果验收
需要每组状态 complete、LoRA 产物齐全、三套测试逐条完整；保存实际环境版本。闭环评测沿用 S6 的 Colab 入口，7B 不转到 Jetson。

In [ ]:
import csv
for group in GROUPS:
    folder = OUT / group
    summary = json.loads((folder / "summary.json").read_text())
    assert summary["status"] == "complete" and summary["training_complete"]
    assert (folder / "lora/adapter_config.json").is_file()
    with (folder / "preds.csv").open(newline="") as f:
        predictions = list(csv.DictReader(f))
    keys = {(r["split"], int(r["idx"])) for r in predictions}
    expected_counts = {"test_seen": 3010, "test_unseen": 2408, "friend_clean": 245}
    expected_keys = {(split, i) for split, count in expected_counts.items() for i in range(count)}
    assert len(predictions) == 5663 and keys == expected_keys, "预测缺失或重复"
    print(group, "完成步数", summary["step"], "预测", len(predictions), "训练秒数", round(summary["elapsed_train_s"], 1))
versions = {k: metadata.version(k) for k in ("torch", "transformers", "peft", "accelerate", "numpy")}
(OUT / "colab_environment.json").write_text(json.dumps({"versions": versions, "gpu": gpu.name,
    "code_archive_sha256": meta["sha256"], "upstream": "17adedef587d5dbc02c167d147692a61179cdf4b"}, indent=2))
print("完整验收通过。权重和预测保存在", OUT)


nolacs 完成步数 208 预测 5663 训练秒数 2502.7
lacs 完成步数 208 预测 5663 训练秒数 2503.5
完整验收通过。权重和预测保存在 /content/drive/MyDrive/COMLLM/s7_runs


In [ ]:
# 严格联合验收：只读检查产物，另存新报告；不加载 GPU 模型。
import hashlib
from pathlib import Path
s7_acceptance = Path("/content/drive/MyDrive/COMLLM/s7_code/s7_acceptance_cell.py")
s7_source = s7_acceptance.read_bytes()
assert hashlib.sha256(s7_source).hexdigest() == "b057077aebf322f99f60ae5c462772490415bba107b976c9c43b55d4ba1bbfd1", "验收入口 SHA 不一致"
exec(compile(s7_source, str(s7_acceptance), "exec"))


严格联合验收通过 /content/drive/MyDrive/COMLLM/s7_runs/acceptance_20261108T103821.435715Z.json {"nolacs": {"step": 208, "predictions": 5663, "lambda_lacs": 0.0, "adapter_tensor_count": 392, "adapter_parameters": 161480704, "metrics": {"test_seen": {"rows": 3010, "search": {"valid_count": 3010, "legal_rate": 1.0, "correct_count": 1046, "accuracy_vs_best": 0.3475083056478405}, "strict": {"valid_count": 3010, "legal_rate": 1.0, "correct_count": 1046, "accuracy_vs_best": 0.3475083056478405}}, "test_unseen": {"rows": 2408, "search": {"valid_count": 2408, "legal_rate": 1.0, "correct_count": 833, "accuracy_vs_best": 0.34593023255813954}, "strict": {"valid_count": 2408, "legal_rate": 1.0, "correct_count": 833, "accuracy_vs_best": 0.34593023255813954}}, "friend_clean": {"rows": 245, "search": {"valid_count": 245, "legal_rate": 1.0, "correct_count": 217, "accuracy_vs_best": 0.8857142857142857}, "strict": {"valid_count": 245, "legal_rate": 1.0, "correct_count": 217, "accuracy_vs_best": 0.8857142857142857

In [ ]:
# SFT-7B训练前基线：与本轮两组使用同样数据、提示词和解码；不训练。
import hashlib, subprocess, sys
from pathlib import Path
EXPECTED_BASELINE_SHA = '0b19cd42fa30e00b6624e067bec4d06e976a68c51cf5c19dfe84bbfb90fb10f8'
baseline_drive = Path('/content/drive/MyDrive/COMLLM/s7_code/s7_sft_baseline_20261108.py')
baseline_local = Path('/content/s7_sft_baseline.py')
payload = baseline_drive.read_bytes()
assert hashlib.sha256(payload).hexdigest() == EXPECTED_BASELINE_SHA
if baseline_local.exists():
    assert baseline_local.read_bytes() == payload
else:
    with baseline_local.open('xb') as stream:
        stream.write(payload)
print('SFT-7B训练前基线：仅推理；预算60分钟；独立输出s7_runs/sft_baseline', flush=True)
baseline_result = subprocess.run([sys.executable, '-u', str(baseline_local), '--max-minutes', '60'], cwd='/content/comllm_s7')
assert baseline_result.returncode == 0, f'基线未完成，返回{baseline_result.returncode}；保留进度，核对上方错误'
print('基线推理完成；随后独立验收预测和指标。', flush=True)


SFT-7B训练前基线：仅推理；预算60分钟；独立输出s7_runs/sft_baseline


AssertionError: 基线未完成，返回1；保留进度，核对上方错误

In [ ]:
# 训练前SFT-7B基线 v2：只修复检查器对默认num_beams=None的误拒，生成口径不变。
import hashlib, subprocess, sys
from pathlib import Path
EXPECTED_BASELINE_SHA = 'b6e0ea1177bee23b06536d9dd3b55c71c7acbc072eea4b045c4a68390dcd1740'
baseline_drive = Path('/content/drive/MyDrive/COMLLM/s7_code/s7_sft_baseline_v2_20261108.py')
baseline_local = Path('/content/s7_sft_baseline_v2.py')
payload = baseline_drive.read_bytes()
assert hashlib.sha256(payload).hexdigest() == EXPECTED_BASELINE_SHA
if baseline_local.exists():
    assert baseline_local.read_bytes() == payload
else:
    with baseline_local.open('xb') as stream:
        stream.write(payload)
print('SFT-7B基线：仅推理，不加载已训LoRA；greedy15/rep1.0/batch4；最多60分钟', flush=True)
with subprocess.Popen([sys.executable, '-u', str(baseline_local), '--max-minutes', '60'], cwd='/content/comllm_s7', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    baseline_returncode = process.wait()
assert baseline_returncode == 0, f'基线未完成，返回{baseline_returncode}；进度保留，核对错误'
print('基线推理完成；随后独立验收5663个键与正式指标。', flush=True)


SFT-7B基线：仅推理，不加载已训LoRA；greedy15/rep1.0/batch4；最多60分钟

Loading weights: 100%|██████████| 339/339 [00:00<00:00, 866.77it/s]
SFT baseline 200/5663 complete rows
SFT baseline 400/5663 complete rows
SFT baseline 600/5663 complete rows
SFT baseline 800/5663 complete rows
SFT baseline 1000/5663 complete rows
SFT baseline 1200/5663 complete rows
SFT baseline 1400/5663 complete rows
SFT baseline 1600/5663 complete rows
SFT baseline 1800/5663 complete rows
SFT baseline 2000/5663 complete rows
SFT baseline 2200/5663 complete rows
SFT baseline 2400/5663 complete rows
SFT baseline 2600/5663 complete rows
SFT baseline 2800/5663 complete rows
SFT baseline 3000/5663 complete rows
SFT baseline 3202/5663 complete rows
SFT baseline 3402/5663 complete rows
SFT baseline 3602/5663 complete rows
SFT baseline 3802/5663 complete rows
SFT baseline 4002/5663 complete rows
SFT baseline 4202/5663 complete rows
SFT baseline 4402/5663 complete rows
SFT baseline 4602/5663 complete rows
SFT baseline 4802/5663 complete

In [ ]:
# 修正版λ=0：原三测试不变；一次8步短测，随后只重载、不追加训练。
import hashlib, os, subprocess, sys
from pathlib import Path
WRAPPER_SHA='a3ab4bb7f56eab7c4a44644f71fcba961fb421e4c721f102dddeddd510ef432d'
FIXED_SHA='31a7a05b849a623931b315ed75d41c822da7638d79c948d41c7603024be57225'
source=Path('/content/drive/MyDrive/COMLLM/s7_fixed_code_20261108/fixed_launch_v2.py')
target=Path('/content/s7_fixed_launch_v2.py'); payload=source.read_bytes()
assert hashlib.sha256(payload).hexdigest()==WRAPPER_SHA
if target.exists():
    assert target.read_bytes()==payload
else:
    with target.open('xb') as f:
        f.write(payload)
env=dict(os.environ); env['PYTHONPATH']='/content/comllm_s7_fixed'; env['PYTHONDONTWRITEBYTECODE']='1'
with subprocess.Popen([sys.executable,'-u',str(target),'--phase','smoke','--fixed-sha',FIXED_SHA,'--max-minutes','120'],cwd='/content',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
    for line in p.stdout:
        print(line,end='',flush=True)
    rc=p.wait()
assert rc==0, f'smoke/reload未通过，返回{rc}；未启动full，保留证据后检查'
print('唯一8步smoke与独立checkpoint reload通过；full仍未启动。',flush=True)


Launching: /usr/bin/python3 -u -m train.s7_train_fixed --base /content/s7_sft7b --s3-data /content/comllm_s7_fixed/data/s3 --s4-data /content/comllm_s7_fixed/data/s4 --out /content/drive/MyDrive/COMLLM/s7_fixed_runs_20261108 --group nolacs --resume --mode smoke --max-steps 8 --max-minutes 30.0

Loading weights: 100%|██████████| 339/339 [00:00<00:00, 835.52it/s]
nolacs step 1/8 loss=0.0
nolacs step 2/8 loss=5.584210157394409e-06
nolacs step 3/8 loss=4.051486644129909e-05
nolacs step 4/8 loss=6.491988824564032e-05
nolacs step 5/8 loss=0.0001733098179101944
nolacs step 6/8 loss=0.0002765040844678879
nolacs step 7/8 loss=0.00037207920104265213
nolacs step 8/8 loss=0.00019543245434761047
{
  "step": 8,
  "elapsed_train_s": 89.65219923799896,
  "training_complete": true,
  "target_steps": 8,
  "status": "smoke_complete",
  "run_dir": "/content/drive/MyDrive/COMLLM/s7_fixed_runs_20261108/smoke/nolacs",
  "expected_eval_rows": 5663
}
Launching: /usr/bin/python3 -u /content/s7_fixed_launch_v2.p

In [8]:
# 修正版λ=0正式训练：同SFT、830条、208更新、原三测试；不继承smoke权重。
import hashlib, os, subprocess, sys
from pathlib import Path
WRAPPER_SHA='a3ab4bb7f56eab7c4a44644f71fcba961fb421e4c721f102dddeddd510ef432d'
FIXED_SHA='31a7a05b849a623931b315ed75d41c822da7638d79c948d41c7603024be57225'
target=Path('/content/s7_fixed_launch_v2.py')
assert hashlib.sha256(target.read_bytes()).hexdigest()==WRAPPER_SHA
env=dict(os.environ); env['PYTHONPATH']='/content/comllm_s7_fixed'; env['PYTHONDONTWRITEBYTECODE']='1'
with subprocess.Popen([sys.executable,'-u',str(target),'--phase','full','--fixed-sha',FIXED_SHA,'--max-minutes','120'],cwd='/content',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
    for line in p.stdout:
        print(line,end='',flush=True)
    rc=p.wait()
assert rc==0, f'full返回{rc}；超时会保存并停止，保留证据后检查'
print('修正版full进程结束；随后独立验收208步、5663预测及adapter。',flush=True)


Launching: /usr/bin/python3 -u -m train.s7_train_fixed --base /content/s7_sft7b --s3-data /content/comllm_s7_fixed/data/s3 --s4-data /content/comllm_s7_fixed/data/s4 --out /content/drive/MyDrive/COMLLM/s7_fixed_runs_20261108 --group nolacs --resume --mode full --max-minutes 119.84314977491667

Loading weights: 100%|██████████| 339/339 [00:00<00:00, 865.38it/s]
nolacs step 1/208 loss=0.0
nolacs step 2/208 loss=6.470829248428345e-06
nolacs step 3/208 loss=3.257662069700018e-05
nolacs step 4/208 loss=7.338733803408104e-05
nolacs step 5/208 loss=0.0002503618597984314
nolacs step 6/208 loss=0.0009310375899076462
nolacs step 7/208 loss=0.0007454119622707367
nolacs step 8/208 loss=0.00038987817606539465
nolacs step 9/208 loss=0.00032956396898953244
nolacs step 10/208 loss=0.00033539051582920365
nolacs step 11/208 loss=0.0002857780400518095
nolacs step 12/208 loss=0.00027846481680171564
nolacs step 13/208 loss=0.00045201589637144934
nolacs step 14/208 loss=0.0005574239403358661
nolacs step 15/